# Intro til Machine Learning — 6: Ekstra opgaver 😈

Velkommen til slutspillet. Disse opgaver er **større, friere og sværere** end dem i
notebook 1-5, og de kombinerer alt, hvad I har lært. Der er ingen fast rækkefølge — vælg
den, der frister mest. Hver opgave har startkode, så I aldrig starter fra en blank side.

| Opgave | Hvad | Bruger |
|---|---|---|
| E.1 | Pokédex-rapporten | pandas + plots |
| E.2 | Gradient descent i 2D | autograd + GD |
| E.3 | Type-oraklet (18 klasser!) | hele ML-pipelinen |
| E.4 | Den store aktiveringsdyst | MNIST + aktiveringer |
| E.5 ⭐ | Hyperparameter-jagten | pandas MØDER PyTorch |
| E.6 | Tegn selv et tal | MNIST + kreativitet |
| E.7 | Fejl-detektiven (boss-level 🐛×5) | ALT |
| E.8 ⭐ | Frit valg fra Kaggle | mod og eventyrlyst |

## Setup (kør altid denne først)

In [ ]:
!pip install -q kagglehub

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from hjaelpefunktioner import vis_mnist_billeder

torch.manual_seed(42)
np.random.seed(42)

# Pokémon
sti = kagglehub.dataset_download("abcsds/pokemon")
df = pd.read_csv(sti + "/Pokemon.csv")

# MNIST (bruges i E.4, E.6 og E.7) — skåret ned som i notebook 5
sti_mnist = kagglehub.dataset_download("oddrationale/mnist-in-csv")
traen_df = pd.read_csv(sti_mnist + "/mnist_train.csv").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv(sti_mnist + "/mnist_test.csv").sample(n=2000, random_state=42).reset_index(drop=True)

X_mnist = torch.tensor(traen_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist = torch.tensor(traen_df["label"].values, dtype=torch.long)
X_mnist_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist_test = torch.tensor(test_df["label"].values, dtype=torch.long)

# 🚨 Plan B hvis Kaggle driller — fjern #'erne:
# df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/Pokemon.csv")
# traen_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_traen_lille.csv.gz")
# test_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_test_lille.csv.gz")

print("klar!", df.shape, X_mnist.shape)

### Opgave E.1 — Pokédex-rapporten 📊

I er blevet hyret som dataanalytikere af Professor Oak. Han vil vide: **hvilken generation
er den "bedste"?** Men "bedst" er jeres valg at definere — stærkest i snit? flest
legendariske? bedst balance mellem angreb og forsvar? mest variation?

**Krav til rapporten (byg den i denne notebook):**
1. Mindst **3 forskellige plots** (fx histogram, scatter, søjler)
2. Mindst én **groupby**
3. Mindst én **ny kolonne**, I selv har beregnet
4. En **konklusion i en markdown-celle**: hvilken generation vinder, og hvorfor?

Startkoden giver et første fingerpeg — resten er op til jer.

In [ ]:
# ET eksempel på en rapport (jeres kan se HELT anderledes ud — det er meningen):

# Plot 1: styrke pr. generation
df.groupby("Generation")["Total"].mean().plot(kind="bar")
plt.ylabel("gennemsnitlig Total")
plt.title("Gen 4 er stærkest i snit")
plt.show()

# Plot 2 + ny kolonne: balance mellem offensiv og defensiv
df["Balance"] = (df["Attack"] + df["Sp. Atk"]) / (df["Defense"] + df["Sp. Def"])
df.groupby("Generation")["Balance"].mean().plot(kind="bar", color="orange")
plt.ylabel("offensiv/defensiv-balance")
plt.title("Gen 1 er mest angrebslysten")
plt.show()

# Plot 3: andel legendariske pr. generation
andel = df.groupby("Generation")["Legendary"].mean()
andel.plot(kind="bar", color="gold")
plt.ylabel("andel legendariske")
plt.title("Gen 3 og 4 har flest legendariske pr. Pokémon")
plt.show()

# Plot 4: spredning — hvor forskellige er generationens Pokémon?
df.boxplot(column="Total", by="Generation")
plt.suptitle("")
plt.title("Spredningen af Total pr. generation")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Eksempel-konklusion: Generation 4 vinder på råstyrke (højeste gennemsnitlige Total ≈ 459) og er sammen med gen 3 rigest på legendariske; gen 2 er svagest i snit. Men pointen med opgaven er PROCESSEN: definér "bedst", mål det, og lad plots bære argumentet. Godkend alt, der opfylder de fire krav og har en begrundet konklusion.</span>

### Opgave E.2 — Gradient descent i 2D 🗺️

I notebook 2 rullede I ned ad en 1D-bakke. Rigtige tabslandskaber har millioner af
dimensioner — men i 2D kan vi stadig TEGNE det. Funktionen
$f(a, b) = (a-1)^2 + 3(b+2)^2$ er en oval dal med bund i $(1, -2)$.

**Jeres mission:** udfyld gradienten og opdateringsskridtene, og tegn ruten ned i dalen
oven på konturplottet. Prøv derefter forskellige startpunkter og læringsrater. (Bonus:
brug autograd i stedet for hånd-gradienten og tjek, at ruten er den samme.)

In [ ]:
def f(a, b):
    return (a - 1) ** 2 + 3 * (b + 2) ** 2

A, B = np.meshgrid(np.linspace(-4, 6, 100), np.linspace(-6, 2, 100))
plt.contour(A, B, f(A, B), levels=30, cmap="viridis")
plt.colorbar(label="f(a, b)")

for start_a, start_b, farve in [(5.0, 1.0, "crimson"), (-3.0, 1.5, "royalblue"), (5.5, -5.5, "green")]:
    a, b = start_a, start_b
    laeringsrate = 0.1
    rute_a, rute_b = [a], [b]
    for i in range(40):
        grad_a = 2 * (a - 1)
        grad_b = 6 * (b + 2)
        a = a - laeringsrate * grad_a
        b = b - laeringsrate * grad_b
        rute_a.append(a)
        rute_b.append(b)
    plt.plot(rute_a, rute_b, "o-", color=farve, markersize=4)

plt.scatter([1], [-2], color="gold", s=150, zorder=3, edgecolors="black", label="minimum")
plt.legend()
plt.title("Tre startpunkter — samme dal")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Bemærk hvordan ruterne bevæger sig hurtigst i b-retningen (koefficienten 3 gør dalen stejlere den vej — gradienten er større) og til sidst kryber langs den flade a-retning. Det er et minieksempel på, hvorfor "forskellige skalaer" (notebook 1!) gør GD kluntet — og hvorfor optimizers som Adam, der skalerer pr. retning, blev opfundet. Autograd-bonus: <code>a = torch.tensor(5.0, requires_grad=True)</code> osv., <code>f(a, b).backward()</code>, og brug <code>a.grad</code>/<code>b.grad</code> — samme rute.</span>

### Opgave E.3 — Type-oraklet 🔮

Kan man se på en Pokémons stats, hvilken **type** den er? Det er klassifikation med
**18 klasser** — jeres hidtil vildeste. Udfyld hullerne (mønstret er 12.6 + notebook 5).

Bagefter: sammenlign med den dovne baseline (gæt altid på den mest almindelige type —
hvor god er DEN? Husk opgave 2.5!), og diskutér: hvorfor er det her SÅ meget sværere end
at spotte legendariske?

In [ ]:
typer = sorted(df["Type 1"].unique())
print(len(typer), "typer:", typer)
type_til_nr = {t: nr for nr, t in enumerate(typer)}

seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = df["Type 1"].map(type_til_nr).values

X_traen_np, X_test_np, y_traen_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_traen, X_test = torch.tensor(X_traen_np), torch.tensor(X_test_np)
y_traen = torch.tensor(y_traen_np, dtype=torch.long)
y_test = torch.tensor(y_test_np, dtype=torch.long)

class TypeOrakel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 64)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(64, 18)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model = TypeOrakel()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoke in range(2000):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X_traen), y_traen)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    gaet = model(X_test).argmax(dim=1)
print(f"oraklets accuracy:  {(gaet == y_test).float().mean().item():.1%}")

mest_almindelige = type_til_nr[df["Type 1"].value_counts().index[0]]   # Water
doven = (y_test == mest_almindelige).float().mean()
print(f"dovne baseline:     {doven.item():.1%} (gæt altid Water)")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Oraklet lander typisk omkring 19-25 % — bedre end den dovne baseline (15 %, andelen af Water i testsættet) og LANGT bedre end rent held (1/18 ≈ 5,6 %), men stadig dårligt. Hvorfor så svært? (1) Stats bestemmer ikke typen — en Water-type kan have alle slags stats; informationen er der simpelthen kun delvist. (2) 18 klasser med meget skæv fordeling (Water 112, Flying 4!). (3) Kun ~35 eksempler pr. klasse i snit. Vigtig lærdom: nogle gange ligger begrænsningen i DATAENE, ikke i modellen — der findes intet netværk, der kan trylle information frem, som ikke er der.</span>

### Opgave E.4 — Den store aktiveringsdyst 🥊

Afgør det én gang for alle (i hvert fald for MNIST): træn det SAMME netværk med **ReLU,
Sigmoid, Tanh og LeakyReLU** og vis de fire test-accuracies i ét søjlediagram. Skabelonen
har allerede løkken — I skal udfylde trænings-rytmen og accuracy-målingen (mønstret fra
notebook 5).

In [ ]:
resultater = {}

for navn, akt in [("ReLU", nn.ReLU()), ("Sigmoid", nn.Sigmoid()),
                  ("Tanh", nn.Tanh()), ("LeakyReLU", nn.LeakyReLU())]:
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(784, 128), akt, nn.Linear(128, 10))
    tabsfunktion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoke in range(3):
        for i in range(0, len(X_mnist), 64):
            optimizer.zero_grad()
            tab = tabsfunktion(model(X_mnist[i:i + 64]), y_mnist[i:i + 64])
            tab.backward()
            optimizer.step()

    with torch.no_grad():
        acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
    resultater[navn] = acc
    print(f"{navn}: {acc:.1%}")

plt.bar(resultater.keys(), resultater.values())
plt.ylim(0.8, 1.0)
plt.ylabel("test-accuracy")
plt.title("Aktiveringsdysten (3 epoker)")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk resultat (3 epoker): ReLU/LeakyReLU/Tanh ligger tæt (~92 %), sigmoid halvandet procentpoint under. Bonus-observation: <code>nn.Sequential</code> er en genvej til små netværk — lagene køres bare i rækkefølge, uden at man selv skriver en klasse. Fint til eksperimenter; klassen giver mere kontrol. (Og bemærk seed-tricket, så alle fire starter med samme vægte — fair dyst!)</span>

### Opgave E.5 ⭐ — Hyperparameter-jagten 🎯

Læringsrate og netværksstørrelse kaldes **hyperparametre** — tal, VI vælger, og som
modellen ikke selv lærer. I praksis prøver man sig systematisk frem. Nu skal pandas møde
PyTorch: kør en **dobbelt for-løkke** over læringsrater × skjulte størrelser (på et udsnit
af MNIST, så det går hurtigt), gem alle resultater i en **DataFrame**, og vis dem som et
**heatmap**. Udfyld hullerne — og find den bedste kombination!

In [ ]:
X_lille = X_mnist[:3000]
y_lille = y_mnist[:3000]

resultater = []
for lr in [0.0001, 0.001, 0.01]:
    for skjulte in [16, 64, 256]:
        torch.manual_seed(42)
        model = nn.Sequential(nn.Linear(784, skjulte), nn.ReLU(), nn.Linear(skjulte, 10))
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        tabsfunktion = nn.CrossEntropyLoss()
        for epoke in range(2):
            for i in range(0, len(X_lille), 64):
                optimizer.zero_grad()
                tab = tabsfunktion(model(X_lille[i:i + 64]), y_lille[i:i + 64])
                tab.backward()
                optimizer.step()

        with torch.no_grad():
            acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
        resultater.append({"lr": lr, "skjulte": skjulte, "accuracy": acc})
        print(f"lr={lr}, skjulte={skjulte}: {acc:.1%}")

res_df = pd.DataFrame(resultater)
tabel = res_df.pivot(index="lr", columns="skjulte", values="accuracy")
print(tabel)

plt.imshow(tabel, cmap="viridis")
plt.colorbar(label="accuracy")
plt.xticks(range(len(tabel.columns)), tabel.columns)
plt.yticks(range(len(tabel.index)), tabel.index)
plt.xlabel("skjulte neuroner")
plt.ylabel("læringsrate")
plt.title("Hyperparameter-jagten")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk vinder lr = 0.01 med 64 skjulte (~91 % på kun 2 epoker og 3.000 billeder), skarpt forfulgt af lr = 0.001 med 256; lr = 0.0001 når slet ikke i mål på så kort træning (helt ned til ~23 % med 16 skjulte!). Det her — en resultat-tabel + et heatmap — er PRÆCIS sådan, rigtige ML-folk vælger hyperparametre (bare med flere kombinationer og mere regnekraft). pandas og PyTorch er et par, ikke konkurrenter.</span>

### Opgave E.6 — Tegn selv et tal ✍️

Kan jeres netværk læse JERES håndskrift? Et 28×28-billede er bare et numpy-array, så I kan
"tegne" med slicing: `billede[raekker, kolonner] = 1.0` maler et hvidt felt. Startkoden
tegner et (meget kantet) 7-tal og spørger modellen.

**Jeres mission:** tegn mindst to andre cifre med slicing, og se om modellen kan læse dem.
Kig på sandsynlighederne — hvornår er den i tvivl? (Og hvis den fejler: er det jeres
håndskrift eller modellens skyld? 😄)

In [ ]:
torch.manual_seed(42)
model_e6 = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
optimizer = torch.optim.Adam(model_e6.parameters(), lr=0.001)
tabsfunktion = nn.CrossEntropyLoss()
for epoke in range(5):
    for i in range(0, len(X_mnist), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model_e6(X_mnist[i:i + 64]), y_mnist[i:i + 64])
        tab.backward()
        optimizer.step()
print("model klar!")

# eksempel: et 1-tal, et 0 og et 4-tal tegnet med slicing
cifre = {}

et_tal = np.zeros((28, 28), dtype=np.float32)
et_tal[4:24, 13:16] = 1.0                 # lodret streg
cifre["1?"] = et_tal

nul = np.zeros((28, 28), dtype=np.float32)
nul[4:7, 9:19] = 1.0                      # top
nul[21:24, 9:19] = 1.0                    # bund
nul[4:24, 7:10] = 1.0                     # venstre
nul[4:24, 18:21] = 1.0                    # højre
cifre["0?"] = nul

fire = np.zeros((28, 28), dtype=np.float32)
fire[4:16, 8:11] = 1.0                    # venstre ben
fire[13:16, 8:21] = 1.0                   # tværstreg
fire[4:24, 17:20] = 1.0                   # højre ben
cifre["4?"] = fire

fig, akser = plt.subplots(1, 3, figsize=(10, 3.5))
for akse, (navn, billede) in zip(akser, cifre.items()):
    with torch.no_grad():
        point = model_e6(torch.tensor(billede.reshape(1, 784)))
    gaet = point.argmax().item()
    tillid = torch.softmax(point, dim=1).max().item()
    akse.imshow(billede, cmap="gray")
    akse.set_title(f"tegnet: {navn}  gæt: {gaet} ({tillid:.0%})")
    akse.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Modellen rammer tit — men ikke altid! Kantede slicing-cifre ligner ikke helt MNIST's blødt håndskrevne (og MNIST-cifre er centreret og har gråtone-kanter), så modellen kan snuble på selv "pæne" tegninger. Det er en vigtig lektion om <b>distribution shift</b>: modeller er bedst på data, der LIGNER træningsdataene. Vil man have det vildere, kan man i Colab uploade et foto af et håndskrevet tal og skalere det ned til 28×28 — spørg en underviser.</span>

### Opgave E.7 — Fejl-detektiven 🕵️ (boss-level)

Pipeline-koden nedenfor skal træne en legendarisk-spotter (som notebook 3) — men der
gemmer sig **FEM fejl** fra hele emnet i den. Nogle crasher med en fejlbesked, andre
snyder i **total stilhed**. Find og ret alle fem!

*Tip: I har mødt hver eneste af fejlene før. Tjek: dataforberedelsen, modellens output,
loop-rytmen (×2) — og til sidst: hvad der egentlig bliver målt.*

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)     # FEJL 1: standardisering manglede (10.9)
y_np = df["Legendary"].astype(int).values.astype("float32")

X_traen_np, X_test_np, y_traen_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_traen, X_test = torch.tensor(X_traen_np), torch.tensor(X_test_np)
y_traen, y_test = torch.tensor(y_traen_np), torch.tensor(y_test_np)

class Spotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()                       # FEJL 2: BCELoss kræver sandsynligheder

    def forward(self, x):
        return self.sigmoid(self.lag2(self.aktivering(self.lag1(x))))

model = Spotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoke in range(500):
    optimizer.zero_grad()                                 # FEJL 3: nulstilling manglede (7.5)
    y_hat = model(X_traen).squeeze()
    tab = tabsfunktion(y_hat, y_traen)
    tab.backward()                                        # FEJL 4: backward SKAL komme før step (10.3)
    optimizer.step()

with torch.no_grad():
    gaet = (model(X_test).squeeze() > 0.5).float()        # FEJL 5: der blev målt på TRÆNINGSdata (10.11)
accuracy = (gaet == y_test).float().mean()
print(f"accuracy: {accuracy.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De fem fejl: (1) ingen standardisering af X — stille fejl, træningen bliver dårlig; (2) ingen sigmoid på outputtet — BCELoss crasher med "all elements of input should be between 0 and 1"; (3) manglende zero_grad — gradienterne hober sig op; (4) step før backward — skridt uden friske gradienter; (5) accuracy målt på træningsdata — måler udenadslære, ikke kunnen. Rettet lander pipelinen på ~93-96 % test-accuracy. Hvis I fandt alle fem: tillykke, I fejlfinder nu på niveau med folk, der har lavet det her i årevis — de laver PRÆCIS de samme fejl.</span>

### Opgave E.8 ⭐ — Frit valg fra Kaggle 🌍

Den sidste boss er friheden selv: **find jeres eget datasæt på
[kaggle.com/datasets](https://www.kaggle.com/datasets)** og træn et netværk på det.

**Tjekliste (= alt hvad I har lært):**
1. 🔍 Find et datasæt med en CSV-fil og mindst én kolonne, der er værd at forudsige.
   Gode begynder-emner: vin-kvalitet, bilpriser, fisk-vægt, film-ratings...
2. 📥 Hent det: `kagglehub.dataset_download("ejer/datasæt-navn")` (navnet står i datasættets URL).
3. 🧹 Udforsk og rens: `head`, `describe`, `isna().sum()` — drop eller udfyld huller,
   og vælg talkolonner som features.
4. 📏 Standardisér features. Train/test-split.
5. 🤔 Regression eller klassifikation? → vælg output-lag og tabsfunktion (opgave 11.8!).
6. 🏋️ Byg model, træn med rytmen, plot tabskurven.
7. 📊 Mål på testsættet — og sammenlign ALTID med en doven baseline (gennemsnittet /
   den største klasse).

Skabelonen nedenfor er jeres startgerüst — udfyld den trin for trin.

In [ ]:
# ET eksempel: forudsig vinkvalitet (regression) fra kemiske målinger
sti_eget = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")
import os
print(os.listdir(sti_eget))

eget_df = pd.read_csv(sti_eget + "/winequality-red.csv")
print(eget_df.shape, "| huller:", eget_df.isna().sum().sum())

features = [k for k in eget_df.columns if k != "quality"]
X_np = eget_df[features].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = eget_df["quality"].values.astype("float32")

X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_tr, X_te = torch.tensor(X_tr), torch.tensor(X_te)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)

model = nn.Sequential(nn.Linear(len(features), 32), nn.ReLU(), nn.Linear(32, 1))
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoke in range(2000):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X_tr).squeeze(), y_tr)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    fejl = (model(X_te).squeeze() - y_te).abs().mean()
    doven = (y_tr.mean() - y_te).abs().mean()      # baseline: gæt altid gennemsnittet
print(f"netværkets gennemsnitsfejl: {fejl.item():.2f} kvalitetspoint")
print(f"dovne baseline:             {doven.item():.2f} kvalitetspoint")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Eksemplet (rødvin) lander typisk på ~0,5 kvalitetspoints fejl mod baselinens ~0,7 — netværket har lært NOGET, men vinkvalitet er svært (mennesker er også uenige!). Til underviseren: alt tæller her, bare tjeklisten følges — de klassiske begynderproblemer er tekstkolonner som features (skal droppes eller oversættes til tal) og glemte huller. Det er meningen, at de skal støde ind i dem og løse dem.</span>

# Det var det hele! 🏁

Hvis I er nået hertil og har løst bare halvdelen: imponerende. I behersker nu hele kæden
fra rå CSV-fil til trænet neuralt netværk — og vigtigere: I ved, hvor fejlene gemmer sig,
og hvornår et flot tal lyver.

Vi ses til næste emne, hvor netværkene lærer at SE. 👁️🧠